# 🪟 Topic 04: PySpark Data Wrangling, Window Functions & Pandas UDFs

## 1. Window Functions in PySpark
Window functions compute calculations across a set of rows related to the current row without collapsing rows (unlike `groupBy`).

Common Window Operations:
- Ranking: `rank()`, `dense_rank()`, `row_number()`
- Analytical: `lead()`, `lag()`, `first()`, `last()`
- Aggregations: `sum()`, `avg()`, `count()` over sliding frame.

---

## 2. PySpark Native Functions vs Python UDFs
> [!WARNING]
> Standard Python UDFs (`@udf`) force JVM $\leftrightarrow$ Python worker serialization overhead.
> Always prefer **PySpark Native Functions (`pyspark.sql.functions`)** or **Pandas UDFs (`@pandas_udf`)** utilizing Apache Arrow zero-copy memory transfer.

---

## 3. Hands-on: Window Functions & Pandas UDFs


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder.master("local[*]").appName("Windowing_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Create sample dataset: Employee Salaries per Department
emp_data = [
    ("Sales", "Alice", 5000),
    ("Sales", "Bob", 4800),
    ("Sales", "Charlie", 5500),
    ("Engineering", "David", 7500),
    ("Engineering", "Eve", 8200),
    ("Engineering", "Frank", 7500),
]

df_emp = spark.createDataFrame(emp_data, ["department", "name", "salary"])

# Define Window Spec: Partition by department, Order by salary descending
window_spec = Window.partitionBy("department").orderBy(F.col("salary").desc())

# Apply Window Analytics: Dense Rank & Salary Difference from Highest Paid in Department
df_windowed = df_emp.withColumn("rank", F.dense_rank().over(window_spec)) \
                    .withColumn("dept_max_salary", F.max("salary").over(Window.partitionBy("department"))) \
                    .withColumn("salary_diff", F.col("dept_max_salary") - F.col("salary"))

print("📊 Window Analytics Results:")
df_windowed.show()
